# Maya's Statistical Journey at BrewMetrics Coffee

> *"Numbers have an important story to tell. They rely on you to give them a clear and convincing voice."* — Stephen Few

Welcome! This notebook is **a story, not a textbook**. Meet **Maya**, a junior data analyst at **BrewMetrics Coffee**, a chain of 50 cafés. Every chapter, Maya faces a real business problem — and statistics is her superpower.

By the end of this notebook you will understand:

| Chapter | Concept | Maya's Question |
|---|---|---|
| 1 | Descriptive Statistics | *"How busy is our café on a typical day?"* |
| 2 | Inferential Statistics | *"Can I trust 30 days of data to tell me about the whole year?"* |
| 3 | Statistical Distributions | *"What shapes does data come in?"* |
| 4 | Normal Distribution | *"Why does the bell curve appear everywhere?"* |
| 5 | Probability | *"What are the chances tomorrow is a busy day?"* |
| 6 | Confidence Intervals | *"How sure am I about my estimate?"* |
| 7 | Correlation | *"Do hot days really sell more iced coffee?"* |

**How to read this notebook**
1. Read the *story* (markdown).
2. Read the *intuition* and the *math*.
3. Run the *code* — change numbers and re-run! Statistics clicks when you experiment.
4. Try the *exercise* at the end of each chapter. Solutions follow it.

Let's begin.

## Setup — Maya's Toolkit

Maya opens her laptop and imports her favourite Python libraries. Run the cell below once before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 5)

rng = np.random.default_rng(seed=42)

print("Toolkit ready! NumPy", np.__version__, "| Pandas", pd.__version__)

## Maya's Dataset — One Year at the Flagship Café

Maya pulls one year (365 days) of data from BrewMetrics' flagship store:

- `customers` — number of customers that day
- `temperature_c` — outside temperature in °C
- `iced_coffee_sales` — number of iced coffees sold
- `hot_coffee_sales` — number of hot coffees sold
- `is_weekend` — 1 if Saturday/Sunday, else 0

We *generate* the data so the notebook is self-contained and reproducible.

In [ ]:
days = pd.date_range("2025-01-01", periods=365, freq="D")

day_of_year = np.arange(365)
seasonal_temp = 18 + 12 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
temperature_c = seasonal_temp + rng.normal(0, 2.5, size=365)

is_weekend = (days.dayofweek >= 5).astype(int)

base_customers = 120 + 25 * is_weekend - 1.2 * (temperature_c - 20)
customers = rng.normal(base_customers, 15).round().astype(int).clip(min=10)

iced_share = 1 / (1 + np.exp(-(temperature_c - 22) / 4))
iced_coffee_sales = rng.binomial(customers, iced_share)
hot_coffee_sales = customers - iced_coffee_sales

cafe = pd.DataFrame({
    "date": days,
    "customers": customers,
    "temperature_c": temperature_c.round(1),
    "iced_coffee_sales": iced_coffee_sales,
    "hot_coffee_sales": hot_coffee_sales,
    "is_weekend": is_weekend,
})

print(f"Maya's dataset: {len(cafe)} days")
cafe.head()

---

# Chapter 1 — Descriptive Statistics

> *"How busy is our café on a typical day?"*

Maya's manager walks in: *"Maya, in one number — how busy are we?"*

Maya could just say "we had 53,000 customers this year" — but that doesn't help him plan staffing for **tomorrow**. He wants a typical day. Welcome to **descriptive statistics** — the art of summarising lots of numbers into a few meaningful ones.

There are three families of summary:

| Family | Question it answers | Examples |
|---|---|---|
| **Central tendency** | Where is the middle? | mean, median, mode |
| **Dispersion** | How spread out is it? | range, variance, std-dev, IQR |
| **Shape** | Is it lopsided or pointy? | skewness, kurtosis |

## 1.1 Central Tendency — "Where is the middle?"

### Mean (the average)

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

Add everything, divide by how many. **Sensitive to outliers** — one freak day with 500 customers will pull the mean up.

### Median (the middle value)

Sort the data, pick the middle number. **Robust to outliers** — that one freak day doesn't move it.

### Mode (the most common value)

The value that occurs most often. Useful for categories (e.g., "most common drink ordered").

> **Maya's rule of thumb**
> - If your data is roughly symmetric → use the **mean**.
> - If your data is skewed or has outliers (incomes, house prices, customer counts on holidays) → use the **median**.

In [ ]:
customers = cafe["customers"]

mean_c = customers.mean()
median_c = customers.median()
mode_c = customers.mode().iloc[0]

print(f"Mean   customers/day : {mean_c:.1f}")
print(f"Median customers/day : {median_c:.1f}")
print(f"Mode   customers/day : {mode_c}")

## 1.2 Dispersion — "How spread out is it?"

Two cafés can have the **same average** (say 120 customers/day) but feel completely different:

- Café A: every day between 115–125 customers. Predictable!
- Café B: some days 50, some days 200. Chaos!

Spread tells us this story.

### Range
$$\text{range} = \max(x) - \min(x)$$
Quick but easily fooled by one outlier.

### Variance — *population* vs *sample*

Average squared distance from the mean. Squaring keeps everything positive and punishes big deviations.

There are **two** formulas — and the difference matters:

**Population variance** (when your data *is* the entire population — divide by **n**):

$$\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \mu)^2$$

**Sample variance** (when your data is a *sample* used to estimate the population — divide by **n − 1**):

$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

> **Why n − 1?** This is called **Bessel's correction**. When we use $\bar{x}$ (computed *from* the sample) in place of the true mean $\mu$, the squared deviations are *systematically a little too small* — because $\bar{x}$ is, by construction, the value closest to all points in the sample. Dividing by $n - 1$ (instead of $n$) compensates for this and makes $s^2$ an **unbiased estimator** of $\sigma^2$.
>
> The number $n - 1$ is called the **degrees of freedom** — once you know $\bar{x}$ and $n - 1$ of the values, the last one is determined.
>
> Rule of thumb: **almost always use $n - 1$** in real life, because you almost always have a sample, not a full population. NumPy / pandas call this `ddof=1`.

### Standard Deviation

$$\sigma = \sqrt{\sigma^2}\quad \text{or}\quad s = \sqrt{s^2}$$

Same units as the data — easier to interpret. *"On a typical day, customer count is roughly ±15 from the average."*

### Interquartile Range (IQR)
$$\text{IQR} = Q_3 - Q_1$$

Middle 50% of the data. **Outlier-resistant**, used in boxplots.

In [ ]:
rng_min, rng_max = customers.min(), customers.max()

var_pop = customers.var(ddof=0)
std_pop = customers.std(ddof=0)

var_sample = customers.var(ddof=1)
std_sample = customers.std(ddof=1)

q1, q3 = customers.quantile([0.25, 0.75])
iqr = q3 - q1

print(f"Range                          : {rng_min} to {rng_max}  (= {rng_max - rng_min})")
print(f"Population variance (÷ n)      : {var_pop:.2f}")
print(f"Sample     variance (÷ n-1)    : {var_sample:.2f}")
print(f"Population std-dev  (÷ n)      : {std_pop:.2f}")
print(f"Sample     std-dev  (÷ n-1)    : {std_sample:.2f}")
print(f"Q1, Q3                         : {q1:.0f}, {q3:.0f}")
print(f"IQR                            : {iqr:.0f}")
print()
print("pandas default for .var() / .std() is ddof=1 (sample).")
print("numpy   default for np.var/np.std is ddof=0 (population). Watch out!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(customers, bins=30, color="#6f4e37", edgecolor="white")
axes[0].axvline(mean_c, color="crimson", lw=2, label=f"Mean = {mean_c:.0f}")
axes[0].axvline(median_c, color="seagreen", lw=2, ls="--", label=f"Median = {median_c:.0f}")
axes[0].set_title("Daily customers — distribution")
axes[0].set_xlabel("Customers per day")
axes[0].set_ylabel("Number of days")
axes[0].legend()

axes[1].boxplot(customers, vert=False, patch_artist=True,
                boxprops=dict(facecolor="#d4a373"))
axes[1].set_title("Boxplot: median, IQR, and outliers")
axes[1].set_xlabel("Customers per day")
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

## 1.3 Shape — "Is it lopsided or pointy?"

- **Skewness** — symmetry. 0 = symmetric, > 0 = long right tail, < 0 = long left tail.
- **Kurtosis** — tailedness / pointiness. Higher = more extreme outliers than a normal curve.

### A one-line summary: `describe()`

Pandas gives you most of this for free.

In [ ]:
print(f"Skewness : {customers.skew():.3f}")
print(f"Kurtosis : {customers.kurt():.3f}")
print()
print("describe():")
print(customers.describe().round(1))

### Try it yourself — Exercise 1

Compute the **mean**, **median**, and **standard deviation** of `temperature_c`. Is the temperature roughly symmetric or skewed?

<details>
<summary><b>Solution (click to expand)</b></summary>

```python
t = cafe["temperature_c"]
print(t.mean(), t.median(), t.std(), t.skew())
```
Mean ≈ median and skewness ≈ 0 → temperature is roughly symmetric.
</details>

---

# Chapter 2 — Inferential Statistics

> *"Maya, can you tell me about ALL 50 cafés? But please don't take a year — by Friday."*

Maya can't survey every café every day. She has to take a **sample** and use it to **infer** something about the whole **population**.

| Term | Meaning | Example |
|---|---|---|
| Population | The full group you care about | All 50 cafés × 365 days = 18,250 day-records |
| Sample | A smaller subset you actually look at | 30 random day-records |
| Parameter | A number describing the population | Population mean μ |
| Statistic | A number computed from the sample | Sample mean x̄ |

Inferential statistics is about answering: **"How well does the sample reflect the truth?"**

## 2.1 Sampling — The honest tasting spoon

Imagine a giant pot of soup (the **population**). You don't drink it all — you stir well and taste a spoonful (the **sample**). If you stir well (random sampling), the spoon represents the pot.

Maya's full year of 365 days is her "population" for this lesson. She takes a random sample of 30 days.

In [ ]:
population = cafe["customers"].values
mu = population.mean()

sample = rng.choice(population, size=30, replace=False)
x_bar = sample.mean()

print(f"Population mean (μ)  : {mu:.2f}")
print(f"Sample mean    (x̄)  : {x_bar:.2f}")
print(f"Difference            : {x_bar - mu:+.2f}")

## 2.2 The Sampling Distribution

What if Maya had taken a *different* random 30 days? She would have got a *different* x̄. So x̄ itself is a random quantity. Its distribution across many possible samples is called the **sampling distribution of the mean**.

Let's simulate: take 5,000 samples of size 30 and look at the histogram of their means.

In [ ]:
n_samples = 5000
sample_size = 30

sample_means = np.array([
    rng.choice(population, size=sample_size, replace=False).mean()
    for _ in range(n_samples)
])

plt.hist(sample_means, bins=40, color="#6f4e37", edgecolor="white")
plt.axvline(mu, color="crimson", lw=2, label=f"Population mean μ = {mu:.1f}")
plt.title(f"Sampling distribution of x̄ (n={sample_size}, {n_samples} samples)")
plt.xlabel("Sample mean (customers/day)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

print(f"Mean of sample means : {sample_means.mean():.2f}  ← very close to μ")
print(f"Std of sample means  : {sample_means.std():.2f}  ← the 'standard error'")

## 2.3 The Central Limit Theorem (CLT) — Statistics' favourite magic trick

**The CLT says:** *Take samples of size n from (almost) any population. As n grows, the distribution of the sample means becomes approximately normal — no matter what the original population looked like.*

Mathematically:

$$\bar{X} \;\sim\; \mathcal{N}\!\left(\mu,\; \frac{\sigma}{\sqrt{n}}\right) \quad \text{for large } n$$

The standard deviation of the sampling distribution has its own name — the **Standard Error**:

$$\text{SE} = \frac{\sigma}{\sqrt{n}}$$

> **Why Maya cares:** Even if daily customer counts are weird and lumpy, the *average* of 30 days behaves like a friendly bell curve. This is what lets her build confidence intervals and test hypotheses — coming next!

Let's prove it: start from a wildly skewed population and watch the magic happen.

In [ ]:
skewed_population = rng.exponential(scale=10, size=100_000)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(skewed_population, bins=50, color="#b5651d", edgecolor="white")
axes[0].set_title("Population\n(very skewed)")

for ax, n in zip(axes[1:], [2, 10, 50]):
    means = np.array([
        rng.choice(skewed_population, size=n, replace=False).mean()
        for _ in range(3000)
    ])
    ax.hist(means, bins=50, color="#6f4e37", edgecolor="white")
    ax.set_title(f"Sample means\n(n = {n})")

plt.suptitle("Central Limit Theorem in action", y=1.02)
plt.tight_layout()
plt.show()

### Try it yourself — Exercise 2

Compute the **standard error** for samples of size n = 30 using the formula `σ / √n`. Compare it with the **empirical** standard deviation of `sample_means` you computed earlier — they should match.

<details>
<summary><b>Solution</b></summary>

```python
sigma = population.std(ddof=0)
se = sigma / np.sqrt(30)
print(f"Theoretical SE : {se:.2f}")
print(f"Empirical SE   : {sample_means.std():.2f}")
```
</details>

---

# Chapter 3 — Statistical Distributions

> *"Not all data looks the same — some are flat, some are bell-shaped, some are spiky."*

A **distribution** is a recipe that tells you which values are likely and which are rare.

Two big families:

| Family | What it describes | Tool |
|---|---|---|
| **Discrete** | Countable outcomes (0, 1, 2, …) | Probability **Mass** Function (PMF) |
| **Continuous** | Measurable outcomes (any real number) | Probability **Density** Function (PDF) |

We'll meet **three** discrete distributions that build on each other:

1. **Bernoulli** — a single yes/no trial.
2. **Binomial** — *n* independent Bernoulli trials. Counts how many "yes"es.
3. **Poisson** — counts of rare events over a fixed window of time or space.

Then we'll preview a continuous distribution to set up Chapter 4.

## 3.1 The Bernoulli Distribution — "Yes or No"

A **Bernoulli trial** has exactly two outcomes:
- **success** (1) with probability **p**
- **failure** (0) with probability **1 − p**

Examples Maya sees every day:
- A customer either buys a muffin (1) or doesn't (0).
- The card payment either succeeds (1) or fails (0).
- A coin flip: heads (1) or tails (0).

### The math

If $X \sim \text{Bernoulli}(p)$:

$$P(X=1) = p \qquad P(X=0) = 1 - p$$

PMF in one line:

$$P(X = x) = p^x (1-p)^{1-x},\quad x \in \{0, 1\}$$

| Quantity | Formula |
|---|---|
| Mean (expected value) | $E[X] = p$ |
| Variance | $\text{Var}(X) = p(1-p)$ |

> **Worked example.** A customer at BrewMetrics buys a muffin with probability *p = 0.30*. So `E[X] = 0.30` muffins per customer, `Var(X) = 0.30 × 0.70 = 0.21`.

In [ ]:
p = 0.30
bern = stats.bernoulli(p)

print(f"Mean (theory)     : {bern.mean():.3f}")
print(f"Variance (theory) : {bern.var():.3f}")
print(f"P(X = 1)          : {bern.pmf(1):.3f}")
print(f"P(X = 0)          : {bern.pmf(0):.3f}")

trials = bern.rvs(size=1000, random_state=rng)
print(f"\nOut of 1000 simulated customers, {trials.sum()} bought a muffin")
print(f"Empirical mean    : {trials.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar([0, 1], [bern.pmf(0), bern.pmf(1)], color=["#cccccc", "#6f4e37"], edgecolor="black")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["No muffin (0)", "Muffin (1)"])
axes[0].set_ylabel("Probability")
axes[0].set_title(f"Bernoulli PMF (p = {p})")
axes[0].set_ylim(0, 1)

axes[1].bar([0, 1], [(trials == 0).mean(), (trials == 1).mean()],
            color=["#cccccc", "#6f4e37"], edgecolor="black")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["No muffin (0)", "Muffin (1)"])
axes[1].set_ylabel("Empirical proportion")
axes[1].set_title("From 1000 simulated customers")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3.2 The Binomial Distribution — "How many out of n?"

A **Binomial** is what you get when you repeat a Bernoulli trial **n** times *independently* and count the number of successes.

Examples Maya cares about:
- Out of 200 customers, how many will buy a muffin?
- Out of 50 card payments, how many will fail?
- Out of 30 latte orders, how many will be made within the 3-minute target?

### The math

If $X \sim \text{Binomial}(n, p)$:

$$P(X = k) = \binom{n}{k}\, p^k (1-p)^{\,n-k},\quad k \in \{0, 1, \dots, n\}$$

The $\binom{n}{k}$ counts the number of ways to choose *which* k of the n trials succeed.

| Quantity | Formula |
|---|---|
| Mean | $E[X] = np$ |
| Variance | $\text{Var}(X) = np(1-p)$ |

> **Bernoulli is just Binomial with n = 1.** Every Binomial is a sum of n independent Bernoullis.

> **Worked example.** A customer buys a muffin with probability *p = 0.30*. Of 200 customers, the expected number of muffin sales is `200 × 0.30 = 60`, with variance `200 × 0.30 × 0.70 = 42` (so std-dev ≈ 6.5).

In [ ]:
n_trials = 200
p_muffin = 0.30
binom = stats.binom(n_trials, p_muffin)

print(f"Mean (theory)     : {binom.mean():.2f}")
print(f"Variance (theory) : {binom.var():.2f}")
print(f"Std-dev (theory)  : {binom.std():.2f}")
print(f"P(X = 60)         : {binom.pmf(60):.4f}")
print(f"P(X ≤ 50)         : {binom.cdf(50):.4f}")
print(f"P(X > 70)         : {1 - binom.cdf(70):.4f}")

simulated = binom.rvs(size=10_000, random_state=rng)
print(f"\nSimulated mean   : {simulated.mean():.2f}")
print(f"Simulated std-dev: {simulated.std():.2f}")

In [ ]:
k = np.arange(0, n_trials + 1)
pmf = binom.pmf(k)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(k, pmf, color="#6f4e37", edgecolor="white")
axes[0].axvline(binom.mean(), color="crimson", lw=2, label=f"Mean = {binom.mean():.0f}")
axes[0].set_xlim(30, 90)
axes[0].set_title(f"Binomial PMF (n={n_trials}, p={p_muffin})")
axes[0].set_xlabel("Number of muffin buyers")
axes[0].set_ylabel("Probability")
axes[0].legend()

for n_, p_, color in [(10, 0.5, "#6f4e37"),
                       (20, 0.5, "#d4a373"),
                       (40, 0.25, "#5b9bd5"),
                       (40, 0.75, "#a0522d")]:
    kk = np.arange(0, n_ + 1)
    axes[1].plot(kk, stats.binom.pmf(kk, n_, p_), "o-", lw=1.5,
                 label=f"n={n_}, p={p_}", color=color)
axes[1].set_title("How n and p reshape the Binomial")
axes[1].set_xlabel("k (successes)")
axes[1].set_ylabel("P(X = k)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3.3 The Poisson Distribution — "How many in a window?"

A **Poisson** distribution counts the number of times something happens in a *fixed window* of time, length, or space — when the events are **rare and independent** and arrive at a steady average rate.

Examples Maya cares about:
- How many customers walk in **per hour**?
- How many complaints does the call centre get **per day**?
- How many typos appear **per page** in the menu?

### The math

If $X \sim \text{Poisson}(\lambda)$ — where $\lambda$ ("lambda") is the *expected* count in the window:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!},\quad k \in \{0, 1, 2, \dots\}$$

| Quantity | Formula |
|---|---|
| Mean | $E[X] = \lambda$ |
| Variance | $\text{Var}(X) = \lambda$  ← **mean = variance** is Poisson's signature! |

### Why does it appear so often?

The Poisson is the **limit of a Binomial** when:
- $n$ is very large (many opportunities for the event), and
- $p$ is very small (each opportunity is unlikely),
- but $np = \lambda$ stays moderate.

Think: every second, a person *might* walk into the café (huge n), but each second the chance is tiny. The total per hour ≈ Poisson(λ).

> **Worked example.** On average, **8 customers walk in per hour** at lunchtime. Then `λ = 8`. The probability that exactly 10 walk in next hour is $P(X=10) = \frac{8^{10} e^{-8}}{10!} \approx 0.099$.

In [ ]:
lam = 8
pois = stats.poisson(lam)

print(f"Mean (theory)     : {pois.mean():.2f}")
print(f"Variance (theory) : {pois.var():.2f}   ← equals the mean!")
print(f"P(X = 10)         : {pois.pmf(10):.4f}")
print(f"P(X ≤ 5)          : {pois.cdf(5):.4f}")
print(f"P(X ≥ 12)         : {1 - pois.cdf(11):.4f}")

simulated = pois.rvs(size=10_000, random_state=rng)
print(f"\nSimulated mean   : {simulated.mean():.2f}")
print(f"Simulated var    : {simulated.var():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

k = np.arange(0, 25)
axes[0].bar(k, pois.pmf(k), color="#6f4e37", edgecolor="white")
axes[0].axvline(lam, color="crimson", lw=2, label=f"λ = {lam}")
axes[0].set_title(f"Poisson PMF (λ = {lam} customers/hour)")
axes[0].set_xlabel("k (customers)")
axes[0].set_ylabel("P(X = k)")
axes[0].legend()

for lam_, color in [(1, "#5b9bd5"), (4, "#6f4e37"), (10, "#d4a373"), (20, "#a0522d")]:
    kk = np.arange(0, 35)
    axes[1].plot(kk, stats.poisson.pmf(kk, lam_), "o-", lw=1.5,
                 label=f"λ = {lam_}", color=color)
axes[1].set_title("As λ grows, the Poisson looks more bell-shaped")
axes[1].set_xlabel("k")
axes[1].set_ylabel("P(X = k)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3.4 When to use which? — A decision guide

| Question | Use |
|---|---|
| "Will *this one* event succeed or fail?" (single yes/no) | **Bernoulli(p)** |
| "Out of *n fixed* trials, how many succeed?" (n known, p known) | **Binomial(n, p)** |
| "How many events happen in a *fixed window*?" (rate known, n undefined / very large) | **Poisson(λ)** |

### The Binomial → Poisson connection (a fun demo)

When `n` is large and `p` is small, **Binomial(n, p) ≈ Poisson(np)**. Maya can verify this.

In [ ]:
n_big, p_small = 1000, 0.008
lam_equiv = n_big * p_small

k = np.arange(0, 25)

plt.bar(k - 0.2, stats.binom.pmf(k, n_big, p_small), width=0.4,
        color="#6f4e37", label=f"Binomial(n={n_big}, p={p_small})")
plt.bar(k + 0.2, stats.poisson.pmf(k, lam_equiv), width=0.4,
        color="#d4a373", label=f"Poisson(λ={lam_equiv:.0f})")
plt.title("Binomial → Poisson when n is large and p is small")
plt.xlabel("k")
plt.ylabel("P(X = k)")
plt.legend()
plt.show()

## 3.5 A peek at continuous distributions

Discrete distributions count things (yes/no, 0/1/2/…). Continuous distributions describe quantities that can take *any* value within a range — like temperature, weight, or time.

The most famous continuous distribution is the **Normal distribution**, which gets its own chapter next. Here is a teaser.

In [ ]:
x = np.linspace(-4, 4, 200)
plt.plot(x, stats.norm.pdf(x), color="#6f4e37", lw=2)
plt.fill_between(x, stats.norm.pdf(x), alpha=0.2, color="#6f4e37")
plt.title("Sneak peek: the Normal distribution PDF")
plt.xlabel("x")
plt.ylabel("density")
plt.show()

### Try it yourself — Exercise 3

**(a) Bernoulli / Binomial.** Suppose 10% of card payments fail (`p = 0.10`). Out of **200** customers, what is the **expected number** of failed payments, and what is the **probability that fewer than 15 fail**?

**(b) Poisson.** A barista breaks an average of **2 cups per shift**. What is the probability she breaks **0 cups** in a shift? **3 or more cups**?

<details>
<summary><b>Solution</b></summary>

**(a)** A single failure is Bernoulli(0.10). For 200 independent customers it's Binomial(200, 0.10):

```python
binom_a = stats.binom(200, 0.10)
print("Expected failures :", binom_a.mean())          # 20
print("Variance          :", binom_a.var())           # 18
print("P(X < 15)         :", binom_a.cdf(14))         # ≈ 0.13
```

**(b)** Cup breakages: Poisson(λ = 2):

```python
pois_b = stats.poisson(2)
print("P(X = 0)  :", pois_b.pmf(0))                   # ≈ 0.135
print("P(X >= 3) :", 1 - pois_b.cdf(2))               # ≈ 0.323
```
</details>

---

# Chapter 4 — The Normal Distribution

> *"The bell curve is everywhere — heights, IQs, measurement errors, even my latte foam thickness."*

The **Normal** (or Gaussian) distribution is the rock star of statistics. Two reasons:

1. It describes **a lot** of natural phenomena.
2. The **Central Limit Theorem** (Chapter 2) makes sample means normal — even when the underlying data isn't.

### Definition

$X \sim \mathcal{N}(\mu, \sigma^2)$ has the bell-shaped PDF

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Two parameters control everything:
- $\mu$ — **mean** (where the peak is)
- $\sigma$ — **standard deviation** (how wide the bell is)

In [ ]:
x = np.linspace(-6, 6, 400)

for mu_, sigma_, label in [(0, 1, "μ=0, σ=1"),
                            (0, 2, "μ=0, σ=2"),
                            (2, 1, "μ=2, σ=1")]:
    plt.plot(x, stats.norm.pdf(x, mu_, sigma_), lw=2, label=label)

plt.title("How μ and σ change the bell")
plt.xlabel("x")
plt.ylabel("density f(x)")
plt.legend()
plt.show()

## 4.1 The Empirical Rule (68 – 95 – 99.7)

For *any* normal distribution, no matter μ or σ:

| Range | Captures |
|---|---|
| μ ± 1σ | ≈ **68%** of the data |
| μ ± 2σ | ≈ **95%** of the data |
| μ ± 3σ | ≈ **99.7%** of the data |

So if BrewMetrics' latte temperature is normally distributed with μ = 65°C and σ = 2°C, then 95% of lattes come out between **61°C and 69°C**. Anything outside ±3σ (below 59°C or above 71°C) is genuinely weird.

In [ ]:
mu_l, sigma_l = 65, 2
x = np.linspace(55, 75, 400)
y = stats.norm.pdf(x, mu_l, sigma_l)

plt.plot(x, y, color="black", lw=2)
for k, color, alpha in [(3, "#dceeff", 0.5), (2, "#a4cdfd", 0.6), (1, "#5b9bd5", 0.7)]:
    mask = (x >= mu_l - k * sigma_l) & (x <= mu_l + k * sigma_l)
    plt.fill_between(x[mask], y[mask], alpha=alpha, color=color, label=f"±{k}σ")

plt.title("Latte temperature — Empirical Rule")
plt.xlabel("Temperature (°C)")
plt.ylabel("density")
plt.legend()
plt.show()

## 4.2 The Z-score — A universal ruler

We can compare data on different scales by **standardising** — subtract the mean, divide by the standard deviation:

$$z = \frac{x - \mu}{\sigma}$$

A z-score tells you *how many standard deviations away from the mean* a value is.
- z = 0 → exactly average
- z = +1.5 → 1.5 standard deviations above average
- z = −2 → 2 standard deviations below average (rare!)

After standardising, any normal becomes the **Standard Normal** $\mathcal{N}(0, 1)$.

> **Maya's question:** A latte came out at 70°C. How unusual is that?
>
> $z = (70 - 65)/2 = 2.5$. Beyond ±2σ → it's in the top 1% of hot lattes!

In [ ]:
latte_temp = 70
z = (latte_temp - mu_l) / sigma_l
prob_hotter = 1 - stats.norm.cdf(z)
print(f"z-score          : {z:.2f}")
print(f"P(temp > 70°C)   : {prob_hotter:.4f}  ({prob_hotter*100:.2f}%)")

### Try it yourself — Exercise 4

A student's exam score is **78** in a class where μ = 70 and σ = 5. What is their z-score, and roughly what percentile are they in?

<details>
<summary><b>Solution</b></summary>

```python
z = (78 - 70) / 5  # = 1.6
percentile = stats.norm.cdf(z) * 100
print(f"z = {z}, percentile ≈ {percentile:.1f}")
```
The student scored about 1.6 standard deviations above average → roughly the **94.5th percentile** (top 5.5% of the class).
</details>

---

# Chapter 5 — Calculating Probability

> *"What are the chances tomorrow is a really busy day?"*

**Probability** is just a number between 0 and 1 that says how likely something is.
- 0 → impossible
- 1 → certain
- 0.5 → coin flip

### The basic rules

Let A and B be events.

| Rule | Formula | Story |
|---|---|---|
| Range | $0 \le P(A) \le 1$ | Probabilities live between 0 and 1 |
| Total | $P(A) + P(\text{not }A) = 1$ | Something must happen |
| Addition (mut. exclusive) | $P(A \cup B) = P(A) + P(B)$ | Pick **one or the other**, never both |
| General addition | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ | Subtract the overlap |
| Multiplication (independent) | $P(A \cap B) = P(A)\,P(B)$ | A doesn't affect B |
| Conditional | $P(A \mid B) = \dfrac{P(A \cap B)}{P(B)}$ | Probability of A *given* B happened |

## 5.1 Probability from data

Maya defines a "busy day" as more than 140 customers. From her year of data:

In [ ]:
busy = cafe["customers"] > 140
weekend = cafe["is_weekend"] == 1

p_busy = busy.mean()
p_weekend = weekend.mean()
p_busy_and_weekend = (busy & weekend).mean()
p_busy_given_weekend = p_busy_and_weekend / p_weekend

print(f"P(busy)               = {p_busy:.3f}")
print(f"P(weekend)            = {p_weekend:.3f}")
print(f"P(busy AND weekend)   = {p_busy_and_weekend:.3f}")
print(f"P(busy | weekend)     = {p_busy_given_weekend:.3f}")
print(f"P(busy | weekday)     = {(busy & ~weekend).sum() / (~weekend).sum():.3f}")

## 5.2 Probability from a distribution

If we *know* (or assume) the distribution, we can compute probabilities exactly using the **CDF** (cumulative distribution function), which gives `P(X ≤ x)`.

Customer count looks roughly normal with the sample mean and std we already computed. Let's estimate **P(more than 150 customers tomorrow)**.

In [ ]:
mu_c = customers.mean()
sigma_c = customers.std(ddof=0)

p_more_than_150 = 1 - stats.norm.cdf(150, mu_c, sigma_c)
p_between_100_140 = stats.norm.cdf(140, mu_c, sigma_c) - stats.norm.cdf(100, mu_c, sigma_c)

print(f"Assuming Normal(μ={mu_c:.1f}, σ={sigma_c:.1f}):")
print(f"  P(customers > 150)       = {p_more_than_150:.3f}")
print(f"  P(100 < customers < 140) = {p_between_100_140:.3f}")

## 5.3 Bayes' Theorem — Updating beliefs with evidence

> *"Before I look outside, I think there's a 30% chance it's raining. Then I see someone walk in with a wet umbrella. Now what should I believe?"*

This is the heart of **Bayes' Theorem**: a formula that tells you how to *update* your beliefs when new evidence arrives.

### The formula

$$\boxed{\; P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)} \;}$$

Each piece has a name and a story:

| Term | Name | What it means |
|---|---|---|
| $P(A)$ | **Prior** | What you believed about A *before* seeing evidence |
| $P(B \mid A)$ | **Likelihood** | If A is true, how likely is the evidence B? |
| $P(B)$ | **Evidence** (marginal) | How likely is the evidence B at all? |
| $P(A \mid B)$ | **Posterior** | Updated belief about A *after* seeing evidence B |

### Where does the formula come from?

From the conditional-probability rule, both directions:

$$P(A \cap B) = P(A \mid B)\, P(B) = P(B \mid A)\, P(A)$$

Equate and divide by $P(B)$ — done.

### The Law of Total Probability (used to compute $P(B)$)

When A is either true or false (and nothing else), $P(B)$ splits into the two ways B can happen:

$$P(B) = P(B \mid A)\, P(A) + P(B \mid \neg A)\, P(\neg A)$$

So Bayes in its full glory:

$$P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B \mid A)\, P(A) + P(B \mid \neg A)\, P(\neg A)}$$

### Worked problem — The faulty espresso machine

BrewMetrics has a quality sensor that flags lattes as "bad" (under-extracted, wrong temperature, etc.).

**The facts:**
- Only **2%** of lattes are actually bad: $P(\text{Bad}) = 0.02$
- The sensor is good but not perfect:
  - If a latte really is bad, the sensor flags it **95%** of the time → $P(\text{Flag} \mid \text{Bad}) = 0.95$
  - If a latte is fine, the sensor still flags it (false alarm) **8%** of the time → $P(\text{Flag} \mid \neg\text{Bad}) = 0.08$

**The question:** The sensor just flagged a latte. **What is the probability it is actually bad?**

Most people guess something like 90%. Bayes will surprise you.

### Step-by-step solution

**Step 1 — Identify each piece**

- Prior: $P(\text{Bad}) = 0.02$, so $P(\neg\text{Bad}) = 0.98$
- Likelihood (true positive): $P(\text{Flag} \mid \text{Bad}) = 0.95$
- Likelihood (false positive): $P(\text{Flag} \mid \neg\text{Bad}) = 0.08$

**Step 2 — Total probability of a flag**

$$P(\text{Flag}) = (0.95)(0.02) + (0.08)(0.98) = 0.019 + 0.0784 = 0.0974$$

**Step 3 — Apply Bayes**

$$P(\text{Bad} \mid \text{Flag}) = \frac{0.95 \times 0.02}{0.0974} \approx 0.195$$

**About 19.5%.** Even after a flag, there's only a ~1-in-5 chance the latte is actually bad. Why? Because bad lattes are rare to begin with (2%), so most of the flags are false alarms.

This is exactly why Bayesian thinking matters in medical testing, fraud detection, spam filters — anywhere the **base rate** is low.

In [ ]:
p_bad = 0.02
p_flag_given_bad = 0.95
p_flag_given_good = 0.08

p_good = 1 - p_bad
p_flag = p_flag_given_bad * p_bad + p_flag_given_good * p_good
p_bad_given_flag = (p_flag_given_bad * p_bad) / p_flag

print(f"P(Bad)              = {p_bad}")
print(f"P(Flag | Bad)       = {p_flag_given_bad}")
print(f"P(Flag | not Bad)   = {p_flag_given_good}")
print(f"P(Flag)             = {p_flag:.4f}")
print(f"P(Bad | Flag)       = {p_bad_given_flag:.4f}  ≈ {p_bad_given_flag*100:.1f}%")

In [ ]:
n_sim = 200_000
is_bad = rng.random(n_sim) < p_bad
flagged = np.where(is_bad,
                   rng.random(n_sim) < p_flag_given_bad,
                   rng.random(n_sim) < p_flag_given_good)

simulated = is_bad[flagged].mean()
print(f"Simulated P(Bad | Flag) over {n_sim:,} lattes : {simulated:.4f}")
print(f"Bayes formula gave                              : {p_bad_given_flag:.4f}")

### Try it yourself — Exercise 5

A disease affects 1 in 1,000 people. A test is 99% accurate (correctly identifies sick people 99% of the time, and correctly identifies healthy people 99% of the time). You test positive. **What is the probability you actually have the disease?**

<details>
<summary><b>Solution</b></summary>

```python
p_d = 1 / 1000
p_pos_d = 0.99
p_pos_nd = 0.01
p_pos = p_pos_d * p_d + p_pos_nd * (1 - p_d)
p_d_pos = (p_pos_d * p_d) / p_pos
print(f"P(disease | positive) = {p_d_pos:.4f}")
```
You'd think 99%, but it's only about **9%** — because the disease is so rare, almost all positives are false alarms. This is the classic *base-rate fallacy*.
</details>

---

# Chapter 6 — Confidence Intervals

> *"How sure am I?"*

Maya estimates the average daily customers from a sample as 132.4. Her boss asks: *"Could it really be 130? Or 140?"* A single number (a **point estimate**) doesn't admit uncertainty. A **confidence interval** does.

### What is a 95% confidence interval?

> *If we repeated the sampling process many times and built a 95% CI each time, about 95 out of 100 of those intervals would contain the true population mean.*

It is **not** "95% probability the mean is in this interval" (the mean is fixed; the *interval* is what's random). This subtlety trips up almost everyone.

### The formula (CI for a mean, when σ is unknown — most common case)

$$\bar{x} \pm t^{*}_{n-1}\cdot \frac{s}{\sqrt{n}}$$

- $\bar{x}$ — sample mean
- $s$ — sample standard deviation
- $n$ — sample size
- $t^{*}_{n-1}$ — critical value from Student's t-distribution with `n-1` degrees of freedom (≈ 1.96 for large n at 95%)

The piece $\dfrac{s}{\sqrt{n}}$ is the **standard error** from Chapter 2.

In [ ]:
sample = rng.choice(population, size=30, replace=False)
n = len(sample)
xbar = sample.mean()
s = sample.std(ddof=1)
se = s / np.sqrt(n)

confidence = 0.95
alpha = 1 - confidence
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)

lo = xbar - t_crit * se
hi = xbar + t_crit * se

print(f"Sample mean (x̄)       : {xbar:.2f}")
print(f"Sample std    (s)     : {s:.2f}")
print(f"Standard error (SE)   : {se:.2f}")
print(f"t* (df={n-1}, 95%)    : {t_crit:.3f}")
print(f"95% CI                : ({lo:.2f}, {hi:.2f})")
print(f"True μ                : {mu:.2f}  ← contained: {lo <= mu <= hi}")

## 6.1 Visualising what "95% confidence" really means

Let's build 100 different 95% CIs from 100 different samples. About 95 should "catch" the true μ. Watch the misses.

In [ ]:
n_intervals = 100
sample_size = 30
caught = 0

plt.figure(figsize=(10, 8))
for i in range(n_intervals):
    s_i = rng.choice(population, size=sample_size, replace=False)
    m = s_i.mean()
    se_i = s_i.std(ddof=1) / np.sqrt(sample_size)
    half = stats.t.ppf(0.975, df=sample_size - 1) * se_i
    lo_i, hi_i = m - half, m + half
    contains = lo_i <= mu <= hi_i
    caught += contains
    color = "#6f4e37" if contains else "crimson"
    plt.plot([lo_i, hi_i], [i, i], color=color, lw=1.5)
    plt.plot(m, i, "o", color=color, markersize=3)

plt.axvline(mu, color="black", lw=2, label=f"True μ = {mu:.1f}")
plt.title(f"100 different 95% CIs — {caught} contained μ")
plt.xlabel("Customers per day")
plt.ylabel("Sample #")
plt.legend()
plt.show()

### Try it yourself — Exercise 6

Build a **99% confidence interval** for the mean from the same `sample` (n = 30). Will it be **wider** or **narrower** than the 95% CI? Why?

<details>
<summary><b>Solution</b></summary>

```python
t99 = stats.t.ppf(0.995, df=n - 1)
lo99 = xbar - t99 * se
hi99 = xbar + t99 * se
print(f"99% CI: ({lo99:.2f}, {hi99:.2f})")
```
**Wider.** To be more confident you'll catch μ, you need a wider net.
</details>

---

# Chapter 7 — Correlation

> *"Do hot days really sell more iced coffee?"*

**Correlation** measures whether two variables move together.
- **Positive** correlation → both rise together (temperature ↑, iced coffee sales ↑).
- **Negative** correlation → one rises while the other falls (temperature ↑, hot coffee sales ↓).
- **Zero** correlation → no linear pattern.

### Pearson correlation coefficient

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2}\sqrt{\sum (y_i - \bar{y})^2}}$$

`r` lives in **[-1, +1]**:
- `+1` → perfect upward straight line
- `0` → no linear relationship
- `-1` → perfect downward straight line

> **Pearson assumes a *linear* relationship.** For monotonic-but-curvy relationships, use **Spearman** (rank-based).

In [ ]:
r_iced, p_iced = stats.pearsonr(cafe["temperature_c"], cafe["iced_coffee_sales"])
r_hot,  p_hot  = stats.pearsonr(cafe["temperature_c"], cafe["hot_coffee_sales"])

print(f"temperature ↔ iced coffee : r = {r_iced:+.3f}  (p = {p_iced:.2e})")
print(f"temperature ↔ hot  coffee : r = {r_hot:+.3f}  (p = {p_hot:.2e})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(cafe["temperature_c"], cafe["iced_coffee_sales"],
                alpha=0.5, color="#4a90e2", s=15)
axes[0].set_title(f"Iced coffee vs temperature  (r = {r_iced:+.2f})")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Iced coffee sales")

axes[1].scatter(cafe["temperature_c"], cafe["hot_coffee_sales"],
                alpha=0.5, color="#a0522d", s=15)
axes[1].set_title(f"Hot coffee vs temperature  (r = {r_hot:+.2f})")
axes[1].set_xlabel("Temperature (°C)")
axes[1].set_ylabel("Hot coffee sales")

plt.tight_layout()
plt.show()

## 7.1 The full correlation matrix (heatmap)

In [ ]:
corr = cafe.drop(columns=["date"]).corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation matrix — BrewMetrics flagship")
plt.tight_layout()
plt.show()

## 7.2 Correlation ≠ Causation

Two famous failures Maya keeps in mind:
- **Ice cream sales** correlate with **drowning deaths**. Does ice cream cause drowning? No — both rise in summer (a *confounder*).
- **Shoe size** correlates with **reading ability** in children. Bigger shoes → better reading? No — both are caused by **age**.

Always ask: *Could a third variable be driving both?*

A correlation of 0 also doesn't mean "no relationship" — it means **no *linear* relationship**.

In [ ]:
x = np.linspace(-3, 3, 200)
y = x ** 2

print(f"Pearson r between x and x² : {stats.pearsonr(x, y)[0]:+.3f}")
plt.scatter(x, y, color="#6f4e37", s=10)
plt.title("Strong relationship, but Pearson r ≈ 0 (it isn't linear)")
plt.xlabel("x")
plt.ylabel("y = x²")
plt.show()

### Try it yourself — Exercise 7

Compute the **Spearman** correlation between `temperature_c` and `iced_coffee_sales`. Is it different from Pearson? Why might it be **higher**?

<details>
<summary><b>Solution</b></summary>

```python
print(stats.spearmanr(cafe["temperature_c"], cafe["iced_coffee_sales"]))
```
Spearman uses **ranks**, so it captures any monotonic (always-up or always-down) relationship — not just straight lines. The iced-coffee/temperature relationship is sigmoid-shaped, so Spearman is usually a bit higher.
</details>

---

# Chapter 8 — Recap & Cheat Sheet

Maya emerges from her week of analysis with answers — and a new skill. Here is everything in one page.

## Concept cheat sheet

| Concept | One-liner | Key formula |
|---|---|---|
| Mean | Where the middle is (sensitive to outliers) | $\bar{x} = \frac{1}{n}\sum x_i$ |
| Median | Middle value (robust) | sort, take middle |
| Population variance | $\sigma^2 = \frac{1}{n}\sum(x_i - \mu)^2$ — when data is the whole population | divide by $n$ |
| Sample variance | $s^2 = \frac{1}{n-1}\sum(x_i - \bar{x})^2$ — Bessel's correction, **unbiased** | divide by $n-1$ |
| Standard deviation | Square root of the variance (same units as data) | $\sigma = \sqrt{\sigma^2},\ s = \sqrt{s^2}$ |
| Standard Error | Spread of the sample mean | $\text{SE} = s / \sqrt{n}$ |
| Central Limit Theorem | Sample means → Normal as n grows | $\bar{X} \sim \mathcal{N}(\mu, \sigma/\sqrt{n})$ |
| Bernoulli | One yes/no trial | $E = p,\ \text{Var} = p(1-p)$ |
| Binomial | Successes in n independent Bernoulli trials | $E = np,\ \text{Var} = np(1-p)$ |
| Poisson | Count of events in a fixed window (rare, independent) | $E = \lambda,\ \text{Var} = \lambda$ |
| Normal distribution | The bell curve | $f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2 / 2\sigma^2}$ |
| Z-score | Distance from mean in std-devs | $z = (x - \mu)/\sigma$ |
| Empirical rule | 68 / 95 / 99.7 | within 1σ / 2σ / 3σ |
| Bayes' Theorem | Update beliefs with evidence | $P(A\mid B) = \frac{P(B\mid A)P(A)}{P(B)}$ |
| 95% Confidence Interval | Range that catches μ in 95% of repeats | $\bar{x} \pm t^{*} \cdot s/\sqrt{n}$ |
| Pearson correlation | Strength of *linear* relationship in [-1, 1] | see Chapter 7 |

## Pandas / SciPy quick reference

```python
df["x"].mean(), df["x"].median()
df["x"].var(ddof=1), df["x"].std(ddof=1)   # sample (default for pandas)
df["x"].var(ddof=0), df["x"].std(ddof=0)   # population
df["x"].describe()

stats.bernoulli(p)            # Bernoulli — one yes/no
stats.binom(n, p)             # Binomial — successes in n trials
stats.poisson(lam)            # Poisson — events per window
stats.norm(mu, sigma)         # Normal distribution
# All have .pmf / .pdf, .cdf, .ppf, .rvs(size, random_state)
stats.norm.cdf(x, mu, sigma)   # P(X ≤ x)
stats.norm.ppf(q, mu, sigma)   # quantile (inverse cdf)

stats.t.interval(0.95, df=n-1, loc=xbar, scale=se)  # CI

stats.pearsonr(x, y)
stats.spearmanr(x, y)
df.corr()
```

## Where to go next
- **Hypothesis testing** (t-tests, chi-square) — using CIs to make yes/no decisions.
- **Linear regression** — turning correlation into prediction.
- **Bayesian inference** — updating prior beliefs with data using Bayes' theorem at scale.
- **Bootstrap & resampling** — getting CIs without assuming normality.

> *"Statistics is the grammar of science."* — Karl Pearson
>
> Now go tell your own data story.